**Navigation** : [Version Python](10g_Speculative_Decoding.ipynb) | [Index](README.md) | [<< Précédent](10e_LLamaSharp_DotNet_BakeOff.ipynb) | [Suivant >>](10h_Roofline_Memory_Bandwidth.ipynb)

# 10g — Décodage spéculatif : le draft qui vérifie

Ce notebook étudie le décodage spéculatif, un autre levier de vitesse après le KV-cache et les mesures TTFT/ITL du notebook 10b.

**Objectifs** : entraîner deux modèles n-grammes, comparer leurs sorties greedy, mesurer les acceptations réelles et le coût modélisé, distinguer le greedy du rejet-échantillonnage sans perte.

**Prérequis** : probabilités discrètes, comptage, dictionnaires et boucles.

**Durée estimée** : 55 minutes.

**Objectifs** : entraîner deux petits modèles n-grammes, comparer leurs prédictions greedy, mesurer l’acceptation et le coût modélisé, puis distinguer cette variante d’une correction d’échantillonnage sans perte.

**Prérequis** : probabilités discrètes, dictionnaires, comptage de fréquences et boucles.

**Durée estimée** : 55 minutes.

L'idée tient en une phrase : un petit modèle **propose** une suite de tokens, et le gros modèle les **vérifie tous d'un coup** — ce que l'autorégression ne permettait pas. Ici le mécanisme est entièrement from scratch en C# avec le BCL .NET 9, mesuré sur CPU, sans API LLM : un bigramme draft et un 4-gramme cible entraînés sur le corpus partagé.


## 1. Le problème : l'autorégression est séquentielle

Générer un texte, c'est produire un token, puis le suivant qui dépend du premier, et ainsi de suite. Chaque token coûte **un passage complet** du modèle cible, et on ne peut pas paralléliser : le token *i+1* n'existe qu'après *i*. Le calcul de la cible est donc le goulot d'étranglement, et la latence de génération (ITL) en est la traduction directe.

Le décodage spéculatif casse cette séquence : au lieu d'attendre un token par passage, on en fait **proposer plusieurs** par un modèle bon marché, puis on les **vérifie en un seul** passage de la cible. Si la cible est d'accord, on a produit plusieurs tokens au prix d'environ un passage.

## 2. Deux vrais modèles n-grammes : le draft et la cible

Pour garder le carnet auto-contenu (CPU, stdlib), le « petit modèle » et le « gros modèle » sont deux vrais modèles probabilistes entraînés from scratch sur le même corpus : un **b-gramme** (contexte d'un mot → rapide, imprécis) et un **4-gramme** (contexte de trois mots → précis, coûteux). Le contenu du corpus n'a pas d'importance pour le mécanisme : il fournit les transitions que les modèles doivent apprendre. Le passage aux vrais LLMs (draft petit / cible 7B) est le même, déclaré `RECOVERABLE-MACHINE` plus bas.

In [1]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text;
using Microsoft.DotNet.Interactive.Formatting;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

sealed class NGramTransition {
    public string Token { get; }
    public int Count { get; set; }
    public NGramTransition(string token) { Token = token; Count = 1; }
}
sealed class NGramModel {
    public int N { get; }
    public List<string> Vocabulary { get; }
    public Dictionary<string,List<NGramTransition>> Counts { get; } = new(StringComparer.Ordinal);
    public NGramModel(int n, List<string> vocabulary) { N=n; Vocabulary=vocabulary.OrderBy(x=>x,StringComparer.Ordinal).ToList(); }
}
record SpeculativeResult(int TokensProduced,int VerifySteps,int Accepted,int ProposedActual,List<string> Completion) {
    public int ProposalBudget(int k)=>VerifySteps*k;
}
record SpecMetrics(double AcceptanceBudget,double AcceptanceActual,double Mu,double Reference,double TotalNew,int VerifySteps,int Accepted,int PaddedProposals,int ActualProposals) {
    public double Speedup(int k,double r)=>Mu/(1.0+k*r);
}
record KOptimalResult(int K,double Speedup);
record DraftNode(string Token,List<DraftNode> Children);
record TreeVerificationResult(List<string> Path,int MatchedTokens,int TokensPerPass,int ProposedNodes,double Alpha);
string ContextKey(IReadOnlyList<string> words,int n) {
    int take=Math.Min(words.Count,Math.Max(0,n-1));
    return string.Join("\u001f",words.Skip(words.Count-take).Take(take));
}
List<string> Tokeniser(string text)=>text.ToLowerInvariant().Split((char[]?)null,StringSplitOptions.RemoveEmptyEntries).ToList();
NGramModel BuildNgram(IReadOnlyList<string> words,int n) {
    var vocabulary=words.Distinct(StringComparer.Ordinal).OrderBy(x=>x,StringComparer.Ordinal).ToList();
    var model=new NGramModel(n,vocabulary);
    for(int i=0;i<=words.Count-n;i++) {
        var context=words.Skip(i).Take(Math.Max(0,n-1)).ToList();
        string key=ContextKey(context,n);
        if(!model.Counts.TryGetValue(key,out var transitions)) {
            transitions=new List<NGramTransition>(); model.Counts[key]=transitions;
        }
        string token=words[i+n-1];
        var existing=transitions.FirstOrDefault(x=>x.Token==token);
        if(existing is null) transitions.Add(new NGramTransition(token)); else existing.Count++;
    }
    return model;
}
string PredictArgmax(NGramModel model,IReadOnlyList<string> context) {
    string key=ContextKey(context,model.N);
    if(!model.Counts.TryGetValue(key,out var transitions)||transitions.Count==0) return model.Vocabulary[0];
    NGramTransition best=transitions[0];
    foreach(var candidate in transitions.Skip(1)) if(candidate.Count>best.Count) best=candidate;
    return best.Token;
}
List<string> GreedyTarget(NGramModel target,IReadOnlyList<string> prompt,int maxNew) {
    if(maxNew<0) throw new ArgumentOutOfRangeException(nameof(maxNew));
    var context=prompt.ToList(); int start=context.Count;
    for(int i=0;i<maxNew;i++) context.Add(PredictArgmax(target,context));
    return context.Skip(start).ToList();
}
SpeculativeResult SpeculativeGreedy(NGramModel target,NGramModel draft,IReadOnlyList<string> prompt,int k,int maxNew) {
    if(k<=0) throw new ArgumentOutOfRangeException(nameof(k),"k doit être strictement positif.");
    if(maxNew<0) throw new ArgumentOutOfRangeException(nameof(maxNew));
    var context=prompt.ToList(); int start=context.Count,newCount=0,steps=0,acceptedTotal=0,actualProposals=0;
    while(newCount<maxNew) {
        int kk=Math.Min(k,maxNew-newCount); actualProposals+=kk;
        var proposed=new List<string>(); var draftContext=context.ToList();
        for(int j=0;j<kk;j++){string token=PredictArgmax(draft,draftContext);proposed.Add(token);draftContext.Add(token);}
        steps++; var targetContext=context.ToList(); int accepted=0; bool rejected=false;
        foreach(string proposal in proposed) {
            string targetToken=PredictArgmax(target,targetContext);
            if(targetToken==proposal){targetContext.Add(targetToken);accepted++;}
            else {
                context.AddRange(proposed.Take(accepted)); context.Add(targetToken);
                newCount+=accepted+1;acceptedTotal+=accepted;rejected=true;break;
            }
        }
        if(!rejected){context.AddRange(proposed);newCount+=kk;acceptedTotal+=kk;}
    }
    return new SpeculativeResult(newCount,steps,acceptedTotal,actualProposals,context.Skip(start).ToList());
}
SpecMetrics MeasureSpeculative(IReadOnlyList<List<string>> prompts,NGramModel target,NGramModel draft,int k,int maxNew) {
    long totalNew=0,totalSteps=0,totalAccepted=0,totalBudget=0,totalActual=0;
    foreach(var prompt in prompts){var result=SpeculativeGreedy(target,draft,prompt,k,maxNew);totalNew+=result.TokensProduced;totalSteps+=result.VerifySteps;totalAccepted+=result.Accepted;totalBudget+=result.ProposalBudget(k);totalActual+=result.ProposedActual;}
    double aBudget=totalBudget==0?0.0:(double)totalAccepted/totalBudget;
    double aActual=totalActual==0?0.0:(double)totalAccepted/totalActual;
    double mu=totalSteps==0?0.0:(double)totalNew/totalSteps;
    return new SpecMetrics(aBudget,aActual,mu,aBudget<1.0?1.0/(1.0-aBudget):double.PositiveInfinity,totalNew,(int)totalSteps,(int)totalAccepted,(int)totalBudget,(int)totalActual);
}
double SpeedupSpec(double mu,int k,double r)=>mu/(1.0+k*r);
Console.WriteLine($"Runtime .NET {Environment.Version} ; fonctions BCL chargées : n-grammes, argmax, cible-greedy et draft-vérifier.");

Runtime .NET 10.0.14 ; fonctions BCL chargées : n-grammes, argmax, cible-greedy et draft-vérifier.


### Lecture des outils chargés

Le runtime .NET et les briques n-grammes sont affichés avant l'entraînement. La cellule suivante entraîne les modèles sur exactement le même corpus que le notebook Python.

In [2]:
var corpusLignes=new string[] {
    "la langue est une suite de mots que le modele essaie de prevoir un a la fois",
    "chaque mot depend des mots qui le precedent et le modele apprend ces dependances",
    "un b-gramme regarde le mot precedent et un 4-gramme regarde les trois precedents",
    "plus le contexte est long plus la prediction est precise",
    "mais plus le contexte est long plus il y a peu de phrases qui le partagent",
    "le petit modele est rapide mais imprecis et le gros modele est precis mais lent",
    "le decodage speculatif utilise les deux a la fois",
    "le petit modele propose une suite de mots et le gros modele les verifie",
    "si les mots sont corrects on en garde plusieurs d un coup",
    "si un mot est faux on s arrete et on reprend le bon",
    "le gain depend de la probabilite que le petit modele ait raison",
    "quand le petit modele ressemble au gros on gagne beaucoup",
    "quand il en est loin on perd le temps des verifications",
    "le decodage speculatif ne change pas la qualite des reponses",
    "il ne fait que produire les memes mots plus vite",
    "c est pour ca qu on l utilise dans les serveurs de grands modeles",
    "le texte que le modele lit est une suite de mots tokenises un a un",
    "chaque token est choisi selon la probabilite que le modele lui donne",
    "la probabilite du prochain mot depend du contexte qui le precede",
    "un contexte long donne une probabilite plus sure mais moins de donnees",
    "un contexte court donne moins de surete mais plus de donnees d entrainement",
    "le decodage speculatif est une technique qui accelere la generation",
    "la technique consiste a proposer des mots puis a les verifier",
    "les mots proposes par le petit modele sont appeles le draft",
    "la verification par le gros modele est un seul passage",
    "un seul passage verifie plusieurs mots a la fois",
    "c est pour ca que le decodage est plus rapide",
    "le taux d acceptation mesure la proportion de mots gardes",
    "un taux d acceptation eleve veut dire un grand gain",
    "un taux d acceptation faible veut dire que le draft est mauvais",
    "le choix de la longueur du draft est important",
    "trop court et on gagne peu et trop long et on gaspille",
    "le ratio de cout entre les deux modeles decide du point optimal",
    "le decodage speculatif est un equilibre entre la qualite du draft et son cout",
    "le petit modele propose une suite de mots et le gros modele les verifie",
    "si les mots sont corrects on en garde plusieurs d un coup",
    "le gain depend de la probabilite que le petit modele ait raison",
    "le decodage speculatif est une technique qui accelere la generation",
    "la verification par le gros modele est un seul passage",
    "le taux d acceptation mesure la proportion de mots gardes",
    "le choix de la longueur du draft est important",
    "le ratio de cout entre les deux modeles decide du point optimal",
    "le decodage speculatif est un equilibre entre la qualite du draft et son cout"
};
var corpusTokens=Tokeniser(string.Join(" ",corpusLignes));
var draftModel=BuildNgram(corpusTokens,2);
var targetModel=BuildNgram(corpusTokens,4);
int uniqueLines=corpusLignes.Distinct(StringComparer.Ordinal).Count();
Console.WriteLine($"Corpus d'entraînement : {corpusLignes.Length} lignes ; {uniqueLines} uniques ; {corpusLignes.Length-uniqueLines} répétées volontairement.");
Console.WriteLine($"Tokens du corpus : {corpusTokens.Count} ; vocabulaire : {draftModel.Vocabulary.Count} mots.");
Console.WriteLine($"Contextes draft 2g : {draftModel.Counts.Count} ; contextes cible 4g : {targetModel.Counts.Count}.")

Corpus d'entraînement : 43 lignes ; 34 uniques ; 9 répétées volontairement.


Tokens du corpus : 506 ; vocabulaire : 154 mots.


Contextes draft 2g : 154 ; contextes cible 4g : 381.


### Lecture de l’entraînement

Le corpus contient bien 43 lignes (34 uniques et 9 reprises), 506 tokens et un vocabulaire de 154 mots. Les 154 contextes du bigramme sont plus courts et plus fréquents que les 381 contextes du 4-gramme ; les sections suivantes comparent leurs prédictions sur ces transitions apprises.

## 3. Le mécanisme draft-vérifier (greedy)

La version mesurée est **greedy** (argmax) : à chaque position, le draft propose le mot qu'il juge le plus probable, et la cible vérifie en comparant **son** argmax. On garde le plus long préfixe où les deux sont d'accord, puis on émet le mot de la cible au point de désaccord. C'est l'essentiel du mécanisme, sans le bruit de l'échantillonnage — la version stochastique fait l'objet de l'exercice 1 : elle accepte avec probabilité `min(1, p_cible(x)/p_draft(x))` puis, au rejet, tire dans le résidu normalisé `max(0, p_cible-p_draft)`. La variante standard ajoute un token bonus cible si toutes les propositions sont acceptées ; ce carnet greedy n'ajoute pas ce bonus.
var greedyBudgetControls=new List<bool>();
foreach(int length in new[]{1,4,8}){var run=SpeculativeGreedy(targetModel,draftModel,promptDemo,length,31);greedyBudgetControls.Add(run.TokensProduced==31&&run.Completion.SequenceEqual(GreedyTarget(targetModel,promptDemo,31));}
var tieBreakModel=new NGramModel(2,new List<string>{"z","a"});tieBreakModel.Counts["x"]=new List<NGramTransition>{new("b"),new("a")};
string insertionTie=PredictArgmax(tieBreakModel,new[]{"x"});
var missingContextModel=new NGramModel(2,new List<string>{"z","a"});string sortedFallback=PredictArgmax(missingContextModel,new[]{"OOV"});
Console.WriteLine($"k=1/4/8, max_new=31 et séquence cible exacte : {greedyBudgetControls.All(x=>x)} ; égalité argmax garde la première transition={insertionTie=="b"}; OOV replie sur le vocabulaire trié={sortedFallback=="a"}.");


In [3]:
var promptDemo=new[]{"la","langue","est"}; int kDemo=4,maxNewDemo=40;
var greedyDemo=SpeculativeGreedy(targetModel,draftModel,promptDemo,kDemo,maxNewDemo);
var baselineDemo=GreedyTarget(targetModel,promptDemo,maxNewDemo);
int paddedDemo=greedyDemo.ProposalBudget(kDemo);
double aBudgetDemo=(double)greedyDemo.Accepted/paddedDemo;
double aActualDemo=(double)greedyDemo.Accepted/greedyDemo.ProposedActual;
Console.WriteLine($"Prompt : {string.Join(" ",promptDemo)} ; k={kDemo} ; maximum demandé={maxNewDemo}.");
Console.WriteLine($"Tokens produits : {greedyDemo.TokensProduced} ; passages cible : {greedyDemo.VerifySteps}.");
Console.WriteLine($"Budget rembourré k×passages : {paddedDemo} ; propositions réellement émises : {greedyDemo.ProposedActual} ; acceptés : {greedyDemo.Accepted}.");
Console.WriteLine($"a_budget={aBudgetDemo:F3} ; a_réel={aActualDemo:F3} ; mu tokens/passage={((double)greedyDemo.TokensProduced/greedyDemo.VerifySteps):F3}.");
Console.WriteLine($"Même séquence que target-greedy : {greedyDemo.Completion.SequenceEqual(baselineDemo)} ; completion={string.Join(" ",greedyDemo.Completion)}.");
var immediateFixtureDraft=new NGramModel(2,new List<string>{"a","b"});
immediateFixtureDraft.Counts["x"] = new List<NGramTransition>{new("a")};
var immediateFixtureTarget=new NGramModel(2,new List<string>{"a","b"});
immediateFixtureTarget.Counts["x"] = new List<NGramTransition>{new("b")};
var immediateRun=SpeculativeGreedy(immediateFixtureTarget,immediateFixtureDraft,new[]{"x"},4,1);
var allAcceptedModel=new NGramModel(2,new List<string>{"a","b"});
allAcceptedModel.Counts["x"] = new List<NGramTransition>{new("a")};
allAcceptedModel.Counts["a"] = new List<NGramTransition>{new("a")};
var allAcceptedRun=SpeculativeGreedy(allAcceptedModel,allAcceptedModel,new[]{"x"},3,3);
bool invalidKRejected=false;try{_ = SpeculativeGreedy(targetModel,draftModel,promptDemo,0,1);}catch(ArgumentOutOfRangeException){invalidKRejected=true;}
var zeroNewRun=SpeculativeGreedy(targetModel,draftModel,promptDemo,4,0);
Console.WriteLine($"Fixtures garde : rejet immédiat accepté={immediateRun.Accepted==0&&immediateRun.TokensProduced==1}; bloc entièrement accepté sans bonus={allAcceptedRun.Accepted==3&&allAcceptedRun.TokensProduced==3}; k=0 rejeté={invalidKRejected}; max_new=0 produit {zeroNewRun.TokensProduced} token.");

Prompt : la langue est ; k=4 ; maximum demandé=40.


Tokens produits : 40 ; passages cible : 11.


Budget rembourré k×passages : 44 ; propositions réellement émises : 42 ; acceptés : 38.


a_budget=0.864 ; a_réel=0.905 ; mu tokens/passage=3.636.


Même séquence que target-greedy : True ; completion=une suite de mots et le gros modele est un seul passage un seul passage un seul passage un seul passage un seul passage un seul passage un seul passage un seul passage un seul passage un seul passage un.


Fixtures garde : rejet immédiat accepté=True; bloc entièrement accepté sans bonus=True; k=0 rejeté=True; max_new=0 produit 0 token.


Une seule invite suffit pour voir le mécanisme : la cellule affiche séparément les 44 places du budget rembourré (11 passages × k=4) et les 42 propositions réellement émises : le dernier bloc n'utilise que deux places. Elle donne aussi les deux fractions acceptées et `mu` mesuré. Dans cette exécution `mu > 1`, là où l'autorégression produirait un token par passage. Cette variante greedy sans bonus émet au plus k tokens par passage.

<a id="exercice-1"></a>

**Exercice 1 — échantillonnage spéculatif sans perte.** Implémente la marginale de sortie d’une étape à partir des distributions discrètes `p_cible` et `p_draft`. Un token proposé `x` est accepté avec probabilité `min(1,p(x)/q(x))`; au rejet, l’échantillon vient du résidu positif `max(0,p-q)` normalisé. La variante standard ajoute un token bonus cible après acceptation de tout le bloc. La garde C# contrôle la masse, la variation totale et deux fixtures où le rejet est possible ; elle n’est pas une estimation Monte-Carlo ni une mesure de latence.

In [4]:
#nullable enable
Dictionary<string,double>? SpeculativeMarginaleEtudiante(
    IReadOnlyDictionary<string,double> cible,IReadOnlyDictionary<string,double> draft) {
    return null; // TODO étudiant : acceptation min(1,p/q), résidu normalisé au rejet.
}
double TotalVariation(IReadOnlyDictionary<string,double> left,IReadOnlyDictionary<string,double> right) {
    var support=left.Keys.Union(right.Keys,StringComparer.Ordinal);
    return 0.5*support.Sum(token=>Math.Abs(left.GetValueOrDefault(token)-right.GetValueOrDefault(token)));
}
bool DistributionValide(IReadOnlyDictionary<string,double>? candidate,IReadOnlyDictionary<string,double> expected) {
    if(candidate is null||candidate.Count!=expected.Count||!candidate.Keys.OrderBy(x=>x,StringComparer.Ordinal).SequenceEqual(expected.Keys.OrderBy(x=>x,StringComparer.Ordinal)))return false;
    if(candidate.Values.Any(x=>!double.IsFinite(x)||x<0.0))return false;
    double mass=candidate.Values.Sum();
    return Math.Abs(mass-1.0)<1e-10&&TotalVariation(candidate,expected)<1e-10;
}
var samplingFixtures=new[]{
    (Name:"rejet sur le token B",P:(IReadOnlyDictionary<string,double>)new Dictionary<string,double>{{"A",0.6},{"B",0.4}},Q:(IReadOnlyDictionary<string,double>)new Dictionary<string,double>{{"A",0.2},{"B",0.8}}),
    (Name:"rejet sur A, résidu en C",P:(IReadOnlyDictionary<string,double>)new Dictionary<string,double>{{"A",0.3},{"B",0.2},{"C",0.5}},Q:(IReadOnlyDictionary<string,double>)new Dictionary<string,double>{{"A",0.6},{"B",0.2},{"C",0.2}})
};
double RejectionMass(IReadOnlyDictionary<string,double> p,IReadOnlyDictionary<string,double> q)=>q.Sum(pair=>pair.Value*Math.Max(0.0,1.0-p.GetValueOrDefault(pair.Key)/pair.Value));
var samplingResults=samplingFixtures.Select(f=>SpeculativeMarginaleEtudiante(f.P,f.Q)).ToArray();
bool samplingAnswerPresent=samplingResults.Any(result=>result is not null);
bool samplingFixturesPass=samplingResults.Length==samplingFixtures.Length&&samplingResults.Select((result,index)=>DistributionValide(result,samplingFixtures[index].P)&&RejectionMass(samplingFixtures[index].P,samplingFixtures[index].Q)>0.0).All(x=>x);
bool samplingMutantsRejected=samplingFixtures.All(f=>!DistributionValide(new Dictionary<string,double>{{f.P.Keys.First(),0.5},{f.P.Keys.Skip(1).First(),0.5}},f.P)&&!DistributionValide(new Dictionary<string,double>{{f.P.Keys.First(),double.NaN}},f.P));
string samplingStatus=!samplingAnswerPresent?"à compléter":samplingFixturesPass?"réponse validée sur les fixtures":"réponse incorrecte sur les fixtures";
Console.WriteLine($"Exercice 1 — marginale stochastique : {samplingStatus}; fixtures non triviales avec rejet={samplingFixtures.Count(f=>RejectionMass(f.P,f.Q)>0.0)}; masses/TV exactes contrôlées={samplingFixturesPass}; constantes/NaN rejetés={samplingMutantsRejected}.");

Exercice 1 — marginale stochastique : à compléter; fixtures non triviales avec rejet=2; masses/TV exactes contrôlées=False; constantes/NaN rejetés=True.


### Lecture du contrôle stochastique

Le stub reste « à compléter ». Les fixtures ont une masse de rejet strictement positive ; la garde n'accepte qu'une distribution finie, de masse 1 et de variation totale nulle par rapport à la marginale cible. Les distributions constantes et NaN sont rejetées.

## 4. Mesurer le taux d'acceptation

On relance la boucle sur les 14 invites prélevées dans le corpus même (un trigramme tous les 3 tokens, cellule suivante) — elles sont donc **dans l'échantillon d'entraînement** : les résultats mesurent ce corpus et ces invites seulement ; ils ne garantissent pas une borne pour des prompts jamais vus. La boucle balaie une gamme de longueurs de draft *k*. Deux grandeurs : `a_budget` divise les tokens acceptés par l'enveloppe rembourrée `verify_steps*k` ; `a_réel` les divise par le nombre réellement proposé, lequel peut être plus petit sur le dernier bloc. `mu` compte les tokens produits par passage. La colonne `1/(1-a_budget)` est conservée comme **repère hypothétique** : elle ne constitue pas une borne démontrée pour ce greedy sans bonus. La formule géométrique suppose une probabilité conditionnelle d'acceptation constante et la variante standard ajoute un token bonus quand tout le draft passe ; ni l'une ni l'autre hypothèse ne décrit exactement les mesures de ce carnet.

In [5]:
var promptOffsets=Enumerable.Range(0,14).Select(i=>i*3).ToArray();
var prompts=promptOffsets.Select(offset=>corpusTokens.Skip(offset).Take(3).ToList()).ToList();
var ks=Enumerable.Range(1,8).ToList();
var res=new Dictionary<int,SpecMetrics>();
foreach(int draftLength in ks)res[draftLength]=MeasureSpeculative(prompts,targetModel,draftModel,draftLength,30);
Console.WriteLine($"Invites d'entraînement : {prompts.Count}, offsets 0..39 par pas de 3 ; max_new=30 ; k=1..8.");
Console.WriteLine(" k | a_budget | a_réel | mu tokens/passage | 1/(1-a_budget), repère");
foreach(int draftLength in ks){var m=res[draftLength];Console.WriteLine($"{draftLength,2} | {m.AcceptanceBudget:F3} | {m.AcceptanceActual:F3} | {m.Mu:F3} | {m.Reference:F3}");}

Invites d'entraînement : 14, offsets 0..39 par pas de 3 ; max_new=30 ; k=1..8.


 k | a_budget | a_réel | mu tokens/passage | 1/(1-a_budget), repère


 1 | 0.590 | 0.590 | 1.000 | 2.442


 2 | 0.498 | 0.504 | 1.687 | 1.992


 3 | 0.374 | 0.385 | 1.900 | 1.598


 4 | 0.328 | 0.343 | 2.222 | 1.488


 5 | 0.265 | 0.282 | 2.246 | 1.361


 6 | 0.222 | 0.241 | 2.258 | 1.286


 7 | 0.193 | 0.213 | 2.283 | 1.238


 8 | 0.169 | 0.190 | 2.295 | 1.204


La lecture est plus subtile qu'elle n'y paraît. Le taux `a_budget` **n'est pas constant** : il décroît de 0,59 (*k*=1) à 0,17 (*k*=8). Plus le draft s'allonge, plus il s'écarte du chemin que la cible aurait suivi — les positions tardives du draft sont acceptées de moins en moins. C'est précisément cette **décroissance** qui fait que `mu` (tokens par passage) croît de façon sous-linéaire puis **sature** vers 2,3 : chaque position de draft ajoutée est acceptée moins souvent, donc rapporte de moins en moins. La colonne `1/(1-a_budget)` est un repère conditionnel qui ne vaut que sous une hypothèse d'acceptation constante et de bonus ; elle ne borne pas cette variante. `a_réel` utilise le vrai nombre de propositions, tandis que `a_budget` garde le dénominateur rembourré du tableau source. C'est le signal qui fixe la longueur de draft optimale.

## 5. Le speedup — et son coût caché

Le gain n'est pas gratuit : chaque passage spéculatif fait **k** calculs de draft (bon marché) **plus** 1 vérification de la cible. On pose le coût d'un passage cible à 1 et le coût d'un token de draft à `r` (ratio). Le speedup face à la cible seule (1 token par passage, coût 1) est alors :

$$\text{speedup} = \frac{\mu}{1 + k \, r}$$

avec `mu` mesuré ci-dessus. Le numérateur est le gain (plus de tokens par passage), le dénominateur le prix (les drafts non acceptés). Le speedup n'est donc **pas monotone** en *k* : il monte, atteint un pic, puis redescend sous 1 quand les drafts coûtent plus qu'ils ne rapportent.

In [6]:
var rs=new[]{0.05,0.10,0.20,0.30};
Console.WriteLine("Speedup modélisé en fonction de k et du coût relatif r du draft");
Console.WriteLine("  k | r=0.05 | r=0.10 | r=0.20 | r=0.30");
foreach(int draftLength in ks){var m=res[draftLength];Console.WriteLine($"{draftLength,3} | "+string.Join(" | ",rs.Select(r=>SpeedupSpec(m.Mu,draftLength,r).ToString("F2",CultureInfo.InvariantCulture))));}

Speedup modélisé en fonction de k et du coût relatif r du draft


  k | r=0.05 | r=0.10 | r=0.20 | r=0.30


  1 | 0.95 | 0.91 | 0.83 | 0.77


  2 | 1.53 | 1.41 | 1.20 | 1.05


  3 | 1.65 | 1.46 | 1.19 | 1.00


  4 | 1.85 | 1.59 | 1.23 | 1.01


  5 | 1.80 | 1.50 | 1.12 | 0.90


  6 | 1.74 | 1.41 | 1.03 | 0.81


  7 | 1.69 | 1.34 | 0.95 | 0.74


  8 | 1.64 | 1.28 | 0.88 | 0.68


Le tableau fait tout le discours. Pour un draft **très bon marché** (`r = 0.05`), le speedup est > 1 dès *k* = 2, culmine à *k* = 4 (1,85) et reste > 1 jusqu'à *k* = 8. Pour un draft **cher** (`r = 0.30`), le gain est marginal : *k* = 2 (1,05) et *k* = 4 (1,01) dépassent à peine 1, *k* = 3 vaut exactement 1,00 (neutre), et tout *k* ≥ 5 est sous 1 — le décodage spéculatif y serait **plus lent** que la cible seule. Le levier décisif n'est pas « aller plus vite » mais **garder le draft à la fois précis et bon marché** : c'est le ratio `r` qui décide si le jeu en vaut la peine.

<a id="exercice-2"></a>

**Exercice 2 — choisir la longueur de draft.** Balaye *k* et maximise `mu(k)/(1+k*r)`. Le callback `mu(k)` doit être évalué pour chaque candidat, et une égalité garde le plus petit *k*. La garde teste la mesure réelle du corpus, une égalité construite, le bord `k_max` et le cas `k_max=1`; aucun chronomètre n’est demandé.

In [7]:
#nullable enable
KOptimalResult? KOptimalEtudiante(Func<int,double> muPourK,double r,int kMax) {
    return null; // TODO étudiant : maximiser mu(k)/(1+k*r), plus petit k en cas d'égalité.
}
KOptimalResult OracleKOptimal(Func<int,double> muPourK,double r,int kMax) {
    if(kMax<1||!double.IsFinite(r)||r<0.0)throw new ArgumentOutOfRangeException();
    int bestK=1;double best=SpeedupSpec(muPourK(1),1,r);
    for(int candidate=2;candidate<=kMax;candidate++){
        double score=SpeedupSpec(muPourK(candidate),candidate,r);
        if(!double.IsFinite(score))throw new InvalidOperationException("mu doit être fini.");
        if(score>best+1e-12){best=score;bestK=candidate;}
    }
    return new KOptimalResult(bestK,best);
}
bool KOptimalAccepte(KOptimalResult? actual,Func<int,double> muForK,double r,int kMax) {
    if(actual is not KOptimalResult result||result.K<1||result.K>kMax||!double.IsFinite(result.Speedup))return false;
    var expected=OracleKOptimal(muForK,r,kMax);
    return result.K==expected.K&&Math.Abs(result.Speedup-expected.Speedup)<1e-10;
}
Func<int,double> measuredMu=k=>MeasureSpeculative(prompts,targetModel,draftModel,k,30).Mu;
var kOptMeasured=KOptimalEtudiante(measuredMu,0.10,12);
bool kOptAnswerPresent=kOptMeasured is not null;
bool measuredFixturePass=KOptimalAccepte(kOptMeasured,measuredMu,0.10,12);
Func<int,double> tieMu=k=>1.0+0.25*k;
var tieKResult=KOptimalEtudiante(tieMu,0.25,3);
bool tieFixturePass=KOptimalAccepte(tieKResult,tieMu,0.25,3)&&tieKResult?.K==1;
Func<int,double> risingMu=k=>k;
var boundaryKResult=KOptimalEtudiante(risingMu,0.05,3);
bool boundaryFixturePass=KOptimalAccepte(boundaryKResult,risingMu,0.05,3)&&boundaryKResult?.K==3;
var singleKResult=KOptimalEtudiante(measuredMu,0.10,1);
bool singleCandidatePass=KOptimalAccepte(singleKResult,measuredMu,0.10,1)&&singleKResult?.K==1;
bool kOptMutantsRejected=!KOptimalAccepte(new KOptimalResult(1,0.0),measuredMu,0.10,12)&&!KOptimalAccepte(new KOptimalResult(4,double.NaN),measuredMu,0.10,12);
bool kOptFixturesPass=measuredFixturePass&&tieFixturePass&&boundaryFixturePass&&singleCandidatePass;
string kOptStatus=!kOptAnswerPresent?"à compléter":kOptFixturesPass?"réponse validée sur les fixtures":"réponse incorrecte sur les fixtures";
Console.WriteLine($"Exercice 2 — choix de k : {kOptStatus}; callback corpus (14 invites, max_new=30), égalité, bord k_max et candidat unique contrôlés={kOptFixturesPass}; constant/NaN rejetés={kOptMutantsRejected}.");

Exercice 2 — choix de k : à compléter; callback corpus (14 invites, max_new=30), égalité, bord k_max et candidat unique contrôlés=False; constant/NaN rejetés=True.


### Lecture du contrôle de k

La garde cherche l'optimum sur les mesures des 14 invites, puis exerce le callback avec une égalité, une frontière et un seul candidat. Elle n'imprime pas la valeur optimale pour laisser la tâche ouverte ; le stub reste « à compléter ».

In [8]:
record SvgFigure(string Content);
string EscapeSvg(string text)=>text.Replace("&","&amp;").Replace("<","&lt;").Replace(">","&gt;").Replace("\"","&quot;").Replace("'","&apos;");
string SvgNum(double value)=>value.ToString("0.###",CultureInfo.InvariantCulture);
string SvgHeat(double value){
    double t=Math.Clamp((value-0.5)/2.0,0.0,1.0);
    (int r,int g,int b) low=(220,75,65),middle=(250,236,120),high=(76,170,103);
    var from=t<0.5?low:middle;var to=t<0.5?middle:high;double u=t<0.5?t*2.0:(t-0.5)*2.0;
    int rr=(int)Math.Round(from.r+(to.r-from.r)*u),gg=(int)Math.Round(from.g+(to.g-from.g)*u),bb=(int)Math.Round(from.b+(to.b-from.b)*u);
    return $"#{rr:X2}{gg:X2}{bb:X2}";
}
string BuildSpeculativeFigure(IReadOnlyList<int> draftLengths,Dictionary<int,SpecMetrics> metrics,IReadOnlyList<double> ratios){
    var svg=new StringBuilder(); const int width=1500,height=450;
    svg.AppendLine($"<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"{width}\" height=\"{height}\" viewBox=\"0 0 {width} {height}\">");
    svg.AppendLine("<rect x=\"0\" y=\"0\" width=\"1500\" height=\"450\" fill=\"#ffffff\"/>");
    svg.AppendLine("<style>text{font-family:Arial,sans-serif;fill:#222} .grid{stroke:#dddddd;stroke-width:1} .axis{stroke:#222;stroke-width:1.4}</style>");
    void Label(double x,double y,string text,int size=12,string anchor="middle")=>svg.AppendLine($"<text x=\"{SvgNum(x)}\" y=\"{SvgNum(y)}\" text-anchor=\"{anchor}\" font-size=\"{size}\">{EscapeSvg(text)}</text>");
    void Line(double x1,double y1,double x2,double y2,string color,int widthPx=1)=>svg.AppendLine($"<line x1=\"{SvgNum(x1)}\" y1=\"{SvgNum(y1)}\" x2=\"{SvgNum(x2)}\" y2=\"{SvgNum(y2)}\" stroke=\"{color}\" stroke-width=\"{widthPx}\"/>");
    void Circle(double x,double y,string color)=>svg.AppendLine($"<circle cx=\"{SvgNum(x)}\" cy=\"{SvgNum(y)}\" r=\"4\" fill=\"{color}\"/>");
    void PanelTitle(int x,string title)=>Label(x,35,title,17);
    PanelTitle(250,"Acceptation : a_budget");PanelTitle(750,"Tokens produits par passage : mu");PanelTitle(1240,"Speedup modélisé : mu / (1 + k r)");
    // Panel 1: acceptance over the padded proposal budget.
    double x0=82,y0=350,w=340,h=260,maxA=0.7;
    for(int tick=0;tick<=7;tick++){double value=tick/10.0,y=y0-h*value/maxA;Line(x0,y,x0+w,y,"#dddddd");Label(x0-10,y+4,value.ToString("F1",CultureInfo.InvariantCulture),11,"end");}
    Line(x0,y0-h,x0,y0,"#222",1);Line(x0,y0,x0+w,y0,"#222",1);
    foreach(int k in draftLengths){double x=x0+w*(k-1)/(draftLengths.Count-1);Label(x,y0+20,k.ToString(CultureInfo.InvariantCulture),11);}
    var points=draftLengths.Select(k=>(X:x0+w*(k-1)/(draftLengths.Count-1),Y:y0-h*metrics[k].AcceptanceBudget/maxA)).ToList();
    for(int i=0;i<points.Count-1;i++)Line(points[i].X,points[i].Y,points[i+1].X,points[i+1].Y,"#b5452a",3);
    foreach(var point in points)Circle(point.X,point.Y,"#b5452a");
    Label(x0+w/2,y0+49,"draft length k");Label(22,y0-h/2,"a_budget",12);
    Line(250,62,278,62,"#b5452a",3);Label(326,66,"accepted / padded budget",11);
    // Panel 2: output tokens per target verification.
    x0=565;y0=350;w=340;h=260;double maxMu=2.5;
    for(int tick=0;tick<=5;tick++){double value=tick*0.5,y=y0-h*value/maxMu;Line(x0,y,x0+w,y,"#dddddd");Label(x0-10,y+4,value.ToString("F1",CultureInfo.InvariantCulture),11,"end");}
    Line(x0,y0-h,x0,y0,"#222",1);Line(x0,y0,x0+w,y0,"#222",1);
    foreach(int k in draftLengths){double x=x0+w*(k-1)/(draftLengths.Count-1);Label(x,y0+20,k.ToString(CultureInfo.InvariantCulture),11);}
    points=draftLengths.Select(k=>(X:x0+w*(k-1)/(draftLengths.Count-1),Y:y0-h*metrics[k].Mu/maxMu)).ToList();
    for(int i=0;i<points.Count-1;i++)Line(points[i].X,points[i].Y,points[i+1].X,points[i+1].Y,"#2a6fb5",3);
    foreach(var point in points)Circle(point.X,point.Y,"#2a6fb5");
    Label(x0+w/2,y0+49,"draft length k");Label(515,y0-h/2,"tokens / passage",12);
    Line(735,62,763,62,"#2a6fb5",3);Label(822,66,"mu measured",11);
    // Panel 3: heatmap for the source's k × r grid.
    double hx=1035,hy=95,cw=58,ch=31;Label(hx+cw*ratios.Count/2,66,"r (draft cost / target cost)",12);
    for(int column=0;column<ratios.Count;column++)Label(hx+cw*(column+0.5),hy-10,ratios[column].ToString("F2",CultureInfo.InvariantCulture),11);
    foreach(int k in draftLengths){int row=k-1;Label(hx-18,hy+ch*(row+0.66),k.ToString(CultureInfo.InvariantCulture),11,"end");for(int column=0;column<ratios.Count;column++){
        double value=SpeedupSpec(metrics[k].Mu,k,ratios[column]);double x=hx+cw*column,y=hy+ch*row;
        svg.AppendLine($"<rect x=\"{SvgNum(x)}\" y=\"{SvgNum(y)}\" width=\"{SvgNum(cw)}\" height=\"{SvgNum(ch)}\" fill=\"{SvgHeat(value)}\" stroke=\"#ffffff\" stroke-width=\"1\"/>");
        Label(x+cw/2,y+ch*0.66,value.ToString("F2",CultureInfo.InvariantCulture),11);
    }}
    Label(hx-28,hy+ch*draftLengths.Count/2,"k",12);
    svg.AppendLine("<defs><linearGradient id=\"speed-scale\" x1=\"0\" y1=\"1\" x2=\"0\" y2=\"0\"><stop offset=\"0%\" stop-color=\"#dc4b41\"/><stop offset=\"50%\" stop-color=\"#faec78\"/><stop offset=\"100%\" stop-color=\"#4caa67\"/></linearGradient></defs>");
    svg.AppendLine("<rect x=\"1335\" y=\"95\" width=\"22\" height=\"248\" fill=\"url(#speed-scale)\" stroke=\"#333\"/>");
    for(int tick=0;tick<=4;tick++){double value=0.5+tick*0.5,y=343-tick*62;Line(1357,y,1363,y,"#222");Label(1368,y+4,value.ToString("F1",CultureInfo.InvariantCulture),10,"start");}
    Label(1346,82,"speedup",10);Label(1240,400,"1.0 = coût cible seul ; >1 gain ; <1 surcoût",11);
    svg.AppendLine("</svg>");return svg.ToString();
}
var svgFigure=BuildSpeculativeFigure(ks,res,rs);
Formatter.Register<SvgFigure>((figure,writer)=>writer.Write(figure.Content),"text/html");
display(new SvgFigure(svgFigure));
Console.WriteLine($"Figure SVG opaque produite : 3 panneaux, {ks.Count} k × {rs.Length} ratios r ; SVG prêt pour validation XML.");

Acceptation : a_budget 
 Tokens produits par passage : mu 
 Speedup modélisé : mu / (1 + k r) 
 
 0.0 
 
 0.1 
 
 0.2 
 
 0.3 
 
 0.4 
 
 0.5 
 
 0.6 
 
 0.7 
 
 
 1 
 2 
 3 
 4 
 5 
 6 
 7 
 8 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 draft length k 
 a_budget 
 
 accepted / padded budget 
 
 0.0 
 
 0.5 
 
 1.0 
 
 1.5 
 
 2.0 
 
 2.5 
 
 
 1 
 2 
 3 
 4 
 5 
 6 
 7 
 8 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 draft length k 
 tokens / passage 
 
 mu measured 
 r (draft cost / target cost) 
 0.05 
 0.10 
 0.20 
 0.30 
 1 
 
 0.95 
 
 0.91 
 
 0.83 
 
 0.77 
 2 
 
 1.53 
 
 1.41 
 
 1.20 
 
 1.05 
 3 
 
 1.65 
 
 1.46 
 
 1.19 
 
 1.00 
 4 
 
 1.85 
 
 1.59 
 
 1.23 
 
 1.01 
 5 
 
 1.80 
 
 1.50 
 
 1.12 
 
 0.90 
 6 
 
 1.74 
 
 1.41 
 
 1.03 
 
 0.81 
 7 
 
 1.69 
 
 1.34 
 
 0.95 
 
 0.74 
 8 
 
 1.64 
 
 1.28 
 
 0.88 
 
 0.68 
 k 
 
 
 
 0.5 
 
 1.0 
 
 1.5 
 
 2.0 
 
 2.5 
 speedup 
 1.0 = coût cible seul ; >1 gain ; <1 surcoût

Figure SVG opaque produite : 3 panneaux, 8 k × 4 ratios r ; SVG prêt pour validation XML.


La figure résume le carnet. À gauche, `a_budget` **décroît** avec *k* — le draft prédit bien le mot suivant, de moins en moins bien les suivants : c'est la fraction calculée sur le budget rembourré. Au centre, `mu` monte puis **sature** vers 2,3, à cause de cette décroissance. À droite, la carte speedup : la zone verte (gain) s'étend vers les *k* grands **quand r est petit**, et se referme vers la zone rouge (perte) dès que le draft coûte cher. C'est exactement le compromis que les systèmes d'inférence réels arbitrent en choisissant le modèle draft.

<a id="exercice-3"></a>

**Exercice 3 — vérifier un draft en arbre.** À partir de l’arbre de nœuds fourni et de la continuation greedy cible, renvoie le chemin candidat qui partage le plus long préfixe avec la cible, le nombre de tokens acceptés, les tokens produits par passage, le nombre de nœuds proposés et `alpha = acceptés / nœuds`. Si le chemin s’arrête avant une divergence, cette variante sans bonus ne compte pas de token supplémentaire ; à la première divergence, le token cible correct est émis. L’arbre de garde contient plusieurs branches et deux continuations contradictoires : le verdict porte sur les chemins et leurs compteurs, pas sur un texte descriptif.

In [9]:
#nullable enable
TreeVerificationResult? SpeculativeTreeEtudiante(DraftNode root,IReadOnlyList<string> cibleGreedy) {
    return null; // TODO étudiant : vérifier les chemins candidats en une passe.
}
List<List<string>> TreePaths(DraftNode root) {
    var result=new List<List<string>>();
    void Visit(DraftNode node,List<string> path){
        var next=path.Append(node.Token).ToList();
        if(node.Children.Count==0){result.Add(next);return;}
        foreach(var child in node.Children)Visit(child,next);
    }
    foreach(var child in root.Children)Visit(child,new List<string>());return result;
}
int CountTreeNodes(DraftNode root)=>1+root.Children.Sum(CountTreeNodes);
int CountDraftNodes(DraftNode root)=>root.Children.Sum(CountTreeNodes);
TreeVerificationResult OracleTree(DraftNode root,IReadOnlyList<string> targetContinuation) {
    var paths=TreePaths(root);List<string> bestPath=paths[0];int bestMatch=-1;
    foreach(var path in paths){int matched=0;while(matched<path.Count&&matched<targetContinuation.Count&&path[matched]==targetContinuation[matched])matched++;if(matched>bestMatch){bestMatch=matched;bestPath=path;}}
    int matchedTokens=Math.Max(0,bestMatch);bool fullLeafAccepted=matchedTokens==bestPath.Count;
    int tokensPerPass=fullLeafAccepted?matchedTokens:matchedTokens+1;
    int proposed=CountDraftNodes(root);double alpha=proposed==0?0.0:(double)matchedTokens/proposed;
    return new TreeVerificationResult(bestPath,matchedTokens,tokensPerPass,proposed,alpha);
}
bool TreeResultAccepte(TreeVerificationResult? actual,DraftNode root,IReadOnlyList<string> targetContinuation) {
    if(actual is not TreeVerificationResult result||result.Path is null||result.Path.Count==0||result.Path.Any(string.IsNullOrWhiteSpace)||!double.IsFinite(result.Alpha))return false;
    var expected=OracleTree(root,targetContinuation);
    return result.Path.SequenceEqual(expected.Path)&&result.MatchedTokens==expected.MatchedTokens&&result.TokensPerPass==expected.TokensPerPass&&result.ProposedNodes==expected.ProposedNodes&&Math.Abs(result.Alpha-expected.Alpha)<1e-10;
}
var branchingTree=new DraftNode("root",new(){
    new DraftNode("A",new(){new DraftNode("B",new()),new DraftNode("C",new())}),
    new DraftNode("X",new(){new DraftNode("D",new())})
});
var treeTargets=new[]{new[]{"A","B","Z"},new[]{"X","Q","Z"}};
var treeResults=treeTargets.Select(targetTokens=>SpeculativeTreeEtudiante(branchingTree,targetTokens)).ToArray();
bool treeAnswerPresent=treeResults.Any(result=>result is not null);
bool treeFixturesPass=treeResults.Length==treeTargets.Length&&treeResults.Select((result,index)=>TreeResultAccepte(result,branchingTree,treeTargets[index])).All(x=>x);
var expectedTree=OracleTree(branchingTree,treeTargets[0]);
bool treeMutantsRejected=!TreeResultAccepte(new TreeVerificationResult(new List<string>{"A","C"},2,2,1,double.NaN),branchingTree,treeTargets[0])&&!TreeResultAccepte(new TreeVerificationResult(new List<string>{"A"},1,1,CountDraftNodes(branchingTree),0.2),branchingTree,treeTargets[0]);
string treeStatus=!treeAnswerPresent?"à compléter":treeFixturesPass?"réponse validée sur les fixtures":"réponse incorrecte sur les fixtures";
Console.WriteLine($"Exercice 3 — draft en arbre : {treeStatus}; arbre branchant, deux chemins cibles et comptage nœuds/chemins contrôlés={treeFixturesPass}; constantes/NaN rejetés={treeMutantsRejected}.");

Exercice 3 — draft en arbre : à compléter; arbre branchant, deux chemins cibles et comptage nœuds/chemins contrôlés=False; constantes/NaN rejetés=True.



warning CS1701: Assuming assembly reference 'Microsoft.AspNetCore.Html.Abstractions, Version=2.3.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' used by 'Microsoft.DotNet.Interactive.Formatting' matches identity 'Microsoft.AspNetCore.Html.Abstractions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' of 'Microsoft.AspNetCore.Html.Abstractions', you may need to supply runtime policy



### Lecture du contrôle arborescent

Le fixture distingue plusieurs chemins et deux continuations cible. Le contrôle compare le chemin, les tokens acceptés et produits, le nombre de nœuds et alpha. Une constante ou une valeur NaN ne peut pas certifier l'exercice.

## 6. Ce que le carnet ne mesure pas

Le mécanisme est **identique** pour de vrais LLMs : le draft y est un petit modèle, la cible un 7B, et le passage de vérification exploite le parallélisme des *k* positions. Ici, draft et cible sont des n-grammes réels entraînés sur CPU — c'est ce qui rend le carnet auto-contenu et les taux d'acceptation **nets** (pas noyés sous la latence réseau d'un endpoint). La mesure du speedup **réel** sur un endpoint de service vLLM authentiﬁé est `RECOVERABLE-MACHINE` : elle demande la machine GPU qui sert vLLM, pas une réécriture du mécanisme. Ordre de grandeur de la littérature, pas un fait mesuré ici : **2-3×** sur T5-XXL (Leviathan, Kalman & Matias 2023, arXiv 2211.17192) et **2-2,5×** sur Chinchilla 70B (Chen et al. 2023, arXiv 2302.01318), copies archivées au gisement.

Deux limites à garder en tête : (1) le speedup mesuré est un **rapport de coût** modélisé (`mu / (1+k r)`), pas un chronomètre — la forme de la courbe et son point de bascule sont robustes, l'ampleur exacte dépend du ratio `r` réel ; (2) l'acceptation greedy ignore le déplacement de distribution que la version stochastique gère (exercice 1).

## 7. Récapitulatif des exercices

1. [Marge stochastique sans perte](#exercice-1) : acceptation, résidu normalisé, masse et variation totale.
2. [Longueur de draft optimale](#exercice-2) : comparer les k avec un callback mu et un coût r.
3. [Vérification d'un arbre draft](#exercice-3) : chemins, branches et compteurs explicites.

Les trois stubs sont répartis dans leurs sections et restent à compléter. Les cellules de contrôle évaluent une réponse lorsqu'elle existe ; elles ne fournissent pas les solutions.

## À retenir

Le corpus contient 43 lignes dont 34 uniques et 9 répétées ; le bigramme draft et le 4-gramme cible produisent une comparaison **greedy** déterministe. Le carnet exécute le prompt `la langue est`, puis balaie 14 prompts, `k=1..8` et les ratios de coût `r=0,05/0,10/0,20/0,30`. Les mesures distinguent l'acceptation rapportée au budget rembourré du nombre de propositions réellement émises. Le coût `mu/(1+k r)` est modélisé ; ce n'est pas la latence d'un service LLM. `1/(1-a_budget)` est un repère hypothétique, pas une borne pour cette variante sans bonus.

Le greedy vérifier-proposer conserve la séquence target-greedy avec les mêmes règles de départage et de repli. Le décodage stochastique standard conserve la distribution cible par acceptation `min(1,p/q)`, résidu normalisé au rejet et token bonus après acceptation complète. La vérification d'un draft en arbre doit compter ses chemins et ses nœuds effectifs.

## Vérifiez votre compréhension

<details><summary>Pourquoi le budget rembourré peut-il dépasser les propositions réelles ?</summary>

Il compte k places à chaque passage, même si le dernier passage ne propose que les tokens qui restent avant `max_new`.
</details>

<details><summary>Le repère `1/(1-a_budget)` garantit-il le débit de ce carnet ?</summary>

Non. Il suppose un taux d'acceptation conditionnel constant et la variante avec token bonus. Le taux agrégé rembourré du greedy mesuré n'établit pas ces hypothèses.
</details>

<details><summary>Le speedup du tableau est-il un temps mesuré de LLM ?</summary>

Non. Il compare le débit mu à un coût de draft modélisé par r ; aucun endpoint LLM n'est appelé.
</details>

## Pour aller plus loin

- [Version Python](10g_Speculative_Decoding.ipynb)
- [Leviathan, Kalman & Matias (2023), *Fast Inference from Transformers via Speculative Decoding*](https://arxiv.org/abs/2211.17192)
- [Chen et al. (2023), *Accelerating Large Language Model Decoding with Speculative Sampling*](https://arxiv.org/abs/2302.01318)
